# Predicción de sobrecostos en contratos públicos (SECOP II)

**Trabajo final · Machine Learning I · Maestría**

**Pregunta:** en el momento de **adjudicar** un proceso de contratación, ¿podemos anticipar si el contrato terminará costando más de lo adjudicado?

**Datos** (datos abiertos, [datos.gov.co](https://www.datos.gov.co)):

| Conjunto | ID | Qué aporta |
|---|---|---|
| [SECOP II - Contratos Electrónicos](https://www.datos.gov.co/Estad-sticas-Nacionales/SECOP-II-Contratos-Electr-nicos/jbjy-vk9h) | `jbjy-vk9h` | el contrato: entidad, proveedor, tipo, **valor del contrato (incluye adiciones)** |
| [SECOP II - Procesos de Contratación](https://www.datos.gov.co/Estad-sticas-Nacionales/SECOP-II-Procesos-de-Contrataci-n/p6dx-8zbt) | `p6dx-8zbt` | el proceso: precio base, **valor adjudicado**, duración, competencia |
| [SECOP II - Adiciones](https://www.datos.gov.co/Gastos-Gubernamentales/SECOP-II-Adiciones/cb9c-h8sn) | `cb9c-h8sn` | solo para **validar** la variable objetivo (sección 4.2) |

**Variable objetivo (sobrecosto):**

$$\text{sobrecosto} = \mathbb{1}\left[\ \texttt{valor\_del\_contrato} > \texttt{valor\_total\_adjudicacion}\ \right]$$

Es decir, el contrato **creció después de adjudicado** (adiciones en valor). En SECOP II `valor_del_contrato` se actualiza con las adiciones; la sección 4 lo verifica de dos formas: con el tope legal de adiciones y con el conjunto de adiciones.

**Regla de oro:** el modelo solo puede usar información **disponible al adjudicar**. Se excluyen identificadores y cualquier columna que se actualice durante la ejecución del contrato (fuga de información).

**Estructura:** 1-3 datos · 4 objetivo y validaciones · 5 variables · 6 partición y validación cruzada por entidad · 7 marco de evaluación · 8 modelos (logística, árbol, bosque aleatorio, gradient boosting) · 9 red neuronal · 10 comparación en prueba · 11 importancia · 12 robustez · 13 fuga · 14 conclusiones.

## 1. Configuración

In [ ]:
import io, time, warnings
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

ANIO = 2022              # año de firma de los contratos (años viejos = contratos ya terminados)
SEMILLA = 42
N_PLIEGUES = 5           # validación cruzada agrupada por entidad, solo dentro de entrenamiento
BETA = 1.0               # umbral: maximiza F-beta en predicciones fuera de pliegue (BETA > 1 prioriza exhaustividad)
CORTE_PRINCIPAL = "legal"   # "legal": tope de la Ley 80 en SMMLV · "nominal": 1,5x en pesos (versión anterior)
TOPE_ERROR = 3.0         # régimen especial (sin tope de Ley 80): crecer más de 3x se trata como error de registro
N_BOOT = 500             # remuestreos bootstrap por entidad para los intervalos de confianza

API = "https://www.datos.gov.co/resource/{}.csv"
CONTRATOS, PROCESOS, ADICIONES = "jbjy-vk9h", "p6dx-8zbt", "cb9c-h8sn"

# Salario mínimo mensual legal vigente, en pesos (2026: Decreto 1469 de 2025).
SMMLV = {2021: 908_526, 2022: 1_000_000, 2023: 1_160_000, 2024: 1_300_000, 2025: 1_423_500, 2026: 1_750_905}

## 2. Descarga de datos (API Socrata)

Se descargan solo las columnas necesarias, paginando de a 50.000 filas.

* **Contratos** firmados en `ANIO`, sin *Contratación directa* ni *Contratación régimen especial* **sin ofertas**. Las variantes *con ofertas* de esas dos modalidades sí entran. La sección 4.4 revisa si las modalidades sin ofertas se pueden incluir.
* **Procesos adjudicados** publicados entre el año anterior y `ANIO` (un contrato firmado en enero pudo publicarse meses antes).

> Los identificadores (`proceso_de_compra`, `id_del_portafolio`, `id_contrato`, `nit_entidad`) se descargan **solo para unir las tablas, validar y separar entrenamiento/prueba por entidad**; nunca entran al modelo.

In [ ]:
COLS_CONTRATOS = [
    # llaves (solo para unir / agrupar / validar)
    "proceso_de_compra", "id_contrato", "nit_entidad",
    # objetivo
    "valor_del_contrato",
    # candidatas (conocidas al adjudicar/firmar)
    "departamento", "orden", "sector", "rama", "entidad_centralizada", "tipo_de_contrato",
    "modalidad_de_contratacion", "codigo_de_categoria_principal", "origen_de_los_recursos",
    "destino_gasto", "es_pyme", "es_grupo", "tipodocproveedor", "habilita_pago_adelantado",
    "el_contrato_puede_ser_prorrogado", "espostconflicto", "obligaci_n_ambiental", "reversion",
    "liquidaci_n", "documentos_tipo", "fecha_de_firma",
    # columnas con fuga: se descargan SOLO para validar y para demostrar el efecto en la sección 13
    "estado_contrato", "dias_adicionados", "presupuesto_general_de_la_nacion_pgn",
    "recursos_propios", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato",
    "valor_facturado", "valor_pagado",
]
COLS_PROCESOS = [
    "id_del_portafolio", "precio_base", "valor_total_adjudicacion", "duracion", "unidad_de_duracion",
    "numero_de_lotes", "proveedores_invitados", "proveedores_con_invitacion",
    "respuestas_al_procedimiento", "proveedores_unicos_con", "fecha_de_ultima_publicaci",
]
SIN_OFERTAS = "('Contratación directa','Contratación régimen especial')"


def _get(url, params):
    """GET con reintentos; una consulta mal formada (400) se reporta de inmediato."""
    for intento in range(5):
        try:
            r = requests.get(url, params=params, timeout=300)
            if r.status_code == 400:
                raise ValueError(f"La API rechazó la consulta: {r.text[:300]}")
            r.raise_for_status()
            return r
        except requests.RequestException:
            time.sleep(2 ** intento)
    raise RuntimeError(f"No se pudo descargar {url} con {params}")


def descargar(dataset, where, columnas, pagina=50_000, max_filas=None):
    """Descarga paginada; max_filas limita el total (para muestras de diagnóstico)."""
    partes, offset = [], 0
    while True:
        limite = pagina if max_filas is None else min(pagina, max_filas - offset)
        params = {"$select": ",".join(columnas), "$where": where, "$order": ":id",
                  "$limit": limite, "$offset": offset}
        df = pd.read_csv(io.StringIO(_get(API.format(dataset), params).text), dtype=str)
        partes.append(df)
        print(f"  {dataset}: offset={offset:>7,}  filas={len(df):,}")
        offset += len(df)
        if len(df) < limite or (max_filas is not None and offset >= max_filas):
            return pd.concat(partes, ignore_index=True)


def descargar_por_ids(dataset, campo, ids, columnas=None, lote=150):
    """Descarga solo las filas cuyas llaves están en `ids`, en lotes (para no exceder el largo de la URL)."""
    ids = list(dict.fromkeys(i for i in ids if isinstance(i, str)))
    partes = []
    for i in range(0, len(ids), lote):
        lista = ",".join("'" + x.replace("'", "''") + "'" for x in ids[i:i + lote])
        params = {"$where": f"{campo} in ({lista})", "$limit": 50_000}
        if columnas:
            params["$select"] = ",".join(columnas)
        partes.append(pd.read_csv(io.StringIO(_get(API.format(dataset), params).text), dtype=str))
    salida = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(columns=columnas or [])
    print(f"  {dataset}: {len(ids):,} llaves consultadas -> {len(salida):,} filas")
    return salida


contratos = descargar(
    CONTRATOS,
    f"fecha_de_firma between '{ANIO}-01-01' and '{ANIO}-12-31T23:59:59' "
    f"and modalidad_de_contratacion not in {SIN_OFERTAS}",
    COLS_CONTRATOS)
procesos = descargar(
    PROCESOS,
    f"fecha_de_publicacion_del between '{ANIO-1}-01-01' and '{ANIO}-12-31T23:59:59' and adjudicado='Si'",
    COLS_PROCESOS)
print("\ncontratos:", contratos.shape, " procesos:", procesos.shape)

## 3. Limpieza y cruce

1. **Procesos** trae varias filas por proceso (una por publicación). Nos quedamos con la última publicación de cada `id_del_portafolio`.
2. Se cruza `Contratos.proceso_de_compra` = `Procesos.id_del_portafolio`.
3. Solo procesos con **un único contrato**: si un proceso tiene varios lotes/contratos, el valor adjudicado total no se puede repartir entre ellos.
4. Se descartan registros con valores no positivos.

In [ ]:
num = lambda s: pd.to_numeric(s, errors="coerce")

contratos = contratos.drop_duplicates("id_contrato")
procesos_u = (procesos.sort_values("fecha_de_ultima_publicaci")
                      .drop_duplicates("id_del_portafolio", keep="last"))
print(f"Procesos: {len(procesos):,} filas -> {len(procesos_u):,} procesos únicos")

n_por_proceso = contratos["proceso_de_compra"].map(contratos["proceso_de_compra"].value_counts())
df = contratos[n_por_proceso == 1].merge(procesos_u, left_on="proceso_de_compra",
                                         right_on="id_del_portafolio", how="inner")
print(f"Contratos: {len(contratos):,} | con un solo contrato por proceso: {(n_por_proceso == 1).sum():,} "
      f"| cruzan con su proceso: {len(df):,}")

df["valor_final"] = num(df["valor_del_contrato"])
df["valor_adjudicado"] = num(df["valor_total_adjudicacion"])
df["precio_base_n"] = num(df["precio_base"])
df = df[(df.valor_final > 0) & (df.valor_adjudicado > 0) & (df.precio_base_n > 0)].reset_index(drop=True)
print(f"Registros finales: {len(df):,}")

## 4. Variable objetivo y validaciones

### 4.1 Objetivo y tope legal de adiciones: dos escenarios

`sobrecosto = valor_del_contrato > valor_total_adjudicacion` (con una tolerancia de 0,01 % por redondeo). `liquidaci_n` **no** interviene en el objetivo: es una variable candidata, que se valida en la sección 4.3.

El artículo 40 de la Ley 80 de 1993 limita las adiciones al 50 % del valor inicial **expresado en salarios mínimos**. En pesos, el tope crece con el salario mínimo del año de la adición:

$$\text{tope} = 1 + 0{,}5 \cdot \frac{\text{SMMLV}_{\text{año de la última adición}}}{\text{SMMLV}_{\text{año de firma}}}$$

La fecha de cada adición no está en la tabla de contratos. Como ninguna adición puede ocurrir después del fin del contrato, se usa el año de `fecha_de_fin_del_contrato` (que se mueve con las prórrogas) como **cota superior**. Esa columna se usa solo para limpiar, nunca como predictor.

| Escenario | Regla de exclusión |
|---|---|
| **nominal** (versión anterior) | crecimiento > 1,5x en pesos, para todas las modalidades |
| **legal** | crecimiento > tope en SMMLV para las modalidades de la Ley 80; régimen especial (sin tope legal): crecimiento > `TOPE_ERROR` |

El modelo principal usa `CORTE_PRINCIPAL`; la sección 12 repite el ajuste con el otro escenario para verificar que las conclusiones se mantienen.

In [ ]:
crecimiento = df.valor_final / df.valor_adjudicado
df["crecimiento"] = crecimiento
df["sobrecosto"] = (crecimiento > 1.0001).astype(int)   # tolerancia por redondeo

print(f"Tasa de sobrecosto antes de limpiar: {df.sobrecosto.mean():.1%}  ({df.sobrecosto.sum():,} de {len(df):,})")
print("\nCrecimiento (valor final / adjudicado) en contratos con sobrecosto:")
print(crecimiento[df.sobrecosto == 1].describe(percentiles=[.1, .5, .9, .99]).round(3))

# tope legal en pesos según el año de la última adición posible
anio_fin = pd.to_datetime(df.fecha_de_fin_del_contrato, errors="coerce").dt.year
anio_ultima = anio_fin.clip(lower=ANIO, upper=max(SMMLV)).fillna(max(SMMLV)).astype(int)
df["tope_legal"] = 1 + 0.5 * anio_ultima.map(SMMLV) / SMMLV[ANIO]
es_especial = df.modalidad_de_contratacion.str.contains("régimen especial", case=False, na=False)

viola = {
    "nominal": crecimiento > 1.5001,
    "legal": np.where(es_especial, crecimiento > TOPE_ERROR, crecimiento > df.tope_legal + 1e-4),
}
print("\nTope legal en pesos según el año de la última adición posible:")
print(df.groupby(anio_ultima).tope_legal.agg(tope="first", contratos="size").round(3))

fig, ax = plt.subplots(figsize=(7, 3.2))
crecimiento[df.sobrecosto == 1].clip(upper=2).hist(bins=80, ax=ax, color="#3b6ea5")
ax.axvline(1.5, color="#c0392b", ls="--", label="1,5x nominal")
for t in sorted(df.tope_legal.round(3).unique()):
    if t > 1.5001:
        ax.axvline(t, color="#e69f00", ls=":", lw=1)
ax.plot([], [], color="#e69f00", ls=":", label="topes legales en SMMLV")
ax.set(title="Crecimiento del valor en contratos con sobrecosto",
       xlabel="valor final / valor adjudicado", ylabel="contratos")
ax.legend(); plt.tight_layout(); plt.show()

ESCENARIOS = {}
for nombre, v in viola.items():
    d = df[~np.asarray(v)].reset_index(drop=True)
    ESCENARIOS[nombre] = d
    print(f"Escenario {nombre:8s}: se descartan {int(np.sum(v)):,} registros -> quedan {len(d):,} "
          f"| tasa de sobrecosto {d.sobrecosto.mean():.1%}")

recuperados = (~np.asarray(viola["legal"])) & np.asarray(viola["nominal"])
print(f"\nContratos que el corte nominal descartaba y el legal conserva: {recuperados.sum():,} "
      f"(crecimiento mediano {crecimiento[recuperados].median():.3f})")

df = ESCENARIOS[CORTE_PRINCIPAL].copy()
print(f"\nEscenario principal: {CORTE_PRINCIPAL} · {len(df):,} contratos · tasa {df.sobrecosto.mean():.1%}")

In [ ]:
print("Tasa de sobrecosto por modalidad:")
display(df.groupby("modalidad_de_contratacion").sobrecosto.agg(tasa="mean", contratos="size")
          .sort_values("contratos", ascending=False).round(3))

### 4.2 Validación del objetivo con el conjunto de adiciones

Si `valor_del_contrato` incluye las adiciones, los contratos marcados con sobrecosto deberían tener registradas **adiciones en valor**, y los demás casi nunca. Se toma una muestra de 300 contratos de cada clase y se buscan sus registros en *SECOP II - Adiciones*. Como los nombres de las columnas de ese conjunto pueden variar, la celda primero las imprime y luego detecta la llave del contrato y el tipo de modificación.

In [ ]:
cabecera = pd.read_csv(io.StringIO(_get(API.format(ADICIONES), {"$limit": 3}).text), dtype=str)
print("Columnas de SECOP II - Adiciones:", cabecera.columns.tolist())
display(cabecera)

campo_id = "id_contrato" if "id_contrato" in cabecera.columns else next(c for c in cabecera.columns if "contrato" in c.lower())
campo_tipo = next((c for c in cabecera.columns if "tipo" in c.lower()), None)
print(f"Llave usada: {campo_id} · columna de tipo: {campo_tipo}")

muestra = pd.concat([df[df.sobrecosto == 1].sample(min(300, int(df.sobrecosto.sum())), random_state=SEMILLA),
                     df[df.sobrecosto == 0].sample(300, random_state=SEMILLA)])
adiciones = descargar_por_ids(ADICIONES, campo_id, muestra.id_contrato.tolist())

if campo_tipo is not None and len(adiciones):
    print("\nTipos de modificación en la muestra:")
    print(adiciones[campo_tipo].value_counts().head(10))
    tipo = adiciones[campo_tipo].fillna("").str.upper()
    en_valor = tipo.str.contains("VALOR") & ~tipo.str.contains("REDUC")
    con_adicion = set(adiciones.loc[en_valor, campo_id])
else:
    con_adicion = set(adiciones[campo_id]) if len(adiciones) else set()
    print("Sin columna de tipo: se cuenta cualquier modificación registrada.")

muestra["tiene_adicion_en_valor"] = muestra.id_contrato.isin(con_adicion).astype(int)
tabla_val = pd.crosstab(muestra.sobrecosto, muestra.tiene_adicion_en_valor, normalize="index").round(3)
print("\nProporción con adición en valor registrada, según la etiqueta:")
display(tabla_val)
sens = muestra.loc[muestra.sobrecosto == 1, "tiene_adicion_en_valor"].mean()
falsa = muestra.loc[muestra.sobrecosto == 0, "tiene_adicion_en_valor"].mean()
print(f"Con sobrecosto y adición registrada: {sens:.1%} · sin sobrecosto y adición registrada: {falsa:.1%}")
print("Lectura: si la primera cifra es alta y la segunda baja, valor_del_contrato sí refleja las adiciones.")

### 4.3 Validación de `liquidaci_n` y de las demás cláusulas del contrato

Una columna que se fija **al firmar** debe tener una distribución parecida en contratos recién firmados y en contratos de `ANIO`. Una que se actualiza durante la ejecución, no: en los contratos nuevos todavía no ha cambiado. Se comparan ambas distribuciones con la **distancia de variación total** (0 = idénticas, 1 = sin nada en común). `estado_contrato` sirve de **control positivo**: se actualiza con seguridad, así que su distancia debe ser grande.

Regla: una cláusula con distancia mayor a 0,10 se trata como sospechosa de actualizarse después de la firma y se excluye del modelo. Es una regla conservadora: una diferencia también puede venir de cambios en las prácticas de contratación entre años.

In [ ]:
CLAUSULAS = ["liquidaci_n", "obligaci_n_ambiental", "reversion", "el_contrato_puede_ser_prorrogado",
             "habilita_pago_adelantado", "documentos_tipo", "espostconflicto"]
desde = (pd.Timestamp.today().normalize() - pd.Timedelta(days=90)).strftime("%Y-%m-%d")
recientes = descargar(
    CONTRATOS, f"fecha_de_firma >= '{desde}' and modalidad_de_contratacion not in {SIN_OFERTAS}",
    CLAUSULAS + ["estado_contrato", "fecha_de_firma"], max_filas=20_000)

def variacion_total(a, b):
    pa = a.fillna("NA").value_counts(normalize=True)
    pb = b.fillna("NA").value_counts(normalize=True)
    return 0.5 * pa.subtract(pb, fill_value=0).abs().sum()

deriva = pd.DataFrame({
    "distancia": {c: variacion_total(df[c], recientes[c]) for c in CLAUSULAS + ["estado_contrato"]},
    f"% 'Si' en {ANIO}": {c: (df[c] == "Si").mean() for c in CLAUSULAS + ["estado_contrato"]},
    "% 'Si' recientes": {c: (recientes[c] == "Si").mean() for c in CLAUSULAS + ["estado_contrato"]},
}).round(3)
display(deriva.sort_values("distancia", ascending=False))

if deriva.loc["estado_contrato", "distancia"] < 0.2:
    print("Aviso: el control positivo cambió poco; la prueba es poco informativa con esta muestra.")
EXCLUIR_POR_DERIVA = [c for c in CLAUSULAS if deriva.loc[c, "distancia"] > 0.10]
print(f"Cláusulas excluidas por sospecha de actualizarse después de firmar: {EXCLUIR_POR_DERIVA or 'ninguna'}")

print("\nTasa de sobrecosto según liquidaci_n:")
display(df.groupby("liquidaci_n").sobrecosto.agg(tasa="mean", contratos="size").round(3))
print("Distribución de liquidaci_n según estado_contrato (filas = estado):")
display(pd.crosstab(df.estado_contrato, df.liquidaci_n, normalize="index").round(3))

### 4.4 ¿Se pueden incluir las modalidades sin ofertas?

*Contratación directa* y *Contratación régimen especial* sin ofertas son la mayoría de los contratos del SECOP II. No hay una razón metodológica para excluirlas del objetivo «predecir sobrecostos en cualquier contrato», pero hay una razón **de datos**: el objetivo necesita un valor adjudicado, y en estas modalidades no siempre hay adjudicación registrada en *Procesos*. Esta celda lo mide con una muestra pseudoaleatoria (procesos cuya llave termina en `00`, cerca del 1 %) y cuenta cuántos contratos hay en total.

In [ ]:
conteo = pd.DataFrame(_get(API.format(CONTRATOS).replace(".csv", ".json"), {
    "$select": "modalidad_de_contratacion, count(*) as contratos",
    "$where": f"fecha_de_firma between '{ANIO}-01-01' and '{ANIO}-12-31T23:59:59'",
    "$group": "modalidad_de_contratacion", "$order": "contratos DESC"}).json())
conteo["contratos"] = num(conteo["contratos"])
print(f"Contratos firmados en {ANIO} por modalidad (todas):")
display(conteo)

so = descargar(
    CONTRATOS,
    f"fecha_de_firma between '{ANIO}-01-01' and '{ANIO}-12-31T23:59:59' "
    f"and modalidad_de_contratacion in {SIN_OFERTAS} and proceso_de_compra like '%00'",
    ["proceso_de_compra", "id_contrato", "modalidad_de_contratacion", "valor_del_contrato", "tipo_de_contrato"],
    max_filas=3_000)
proc_so = descargar_por_ids(PROCESOS, "id_del_portafolio", so.proceso_de_compra,
                            ["id_del_portafolio", "adjudicado", "valor_total_adjudicacion", "precio_base"])
proc_so = proc_so.drop_duplicates("id_del_portafolio", keep="last")
diag = so.merge(proc_so, left_on="proceso_de_compra", right_on="id_del_portafolio", how="left")
diag["tiene_proceso"] = diag.id_del_portafolio.notna()
diag["tiene_adjudicado"] = num(diag.valor_total_adjudicacion) > 0
diag["crece"] = num(diag.valor_del_contrato) > num(diag.valor_total_adjudicacion) * 1.0001

resumen_so = diag.groupby("modalidad_de_contratacion").agg(
    contratos=("id_contrato", "size"), con_proceso=("tiene_proceso", "mean"),
    con_valor_adjudicado=("tiene_adjudicado", "mean"),
    tasa_sobrecosto_si_hay_valor=("crece", lambda s: s[diag.loc[s.index, "tiene_adjudicado"]].mean()))
display(resumen_so.round(3))
print("Tipo de contrato en la muestra:")
display(so.tipo_de_contrato.value_counts(normalize=True).head(6).round(3))

## 5. Selección de variables: identificadores y fuga de información

### 5.1 Columnas excluidas

**Identificadores** (únicos por registro o por entidad/persona; no generalizan y permiten memorizar):
`id_contrato`, `referencia_del_contrato`, `proceso_de_compra`, `id_del_proceso`, `id_del_portafolio`, `referencia_del_proceso`, `id_adjudicacion`, `urlproceso`, `nit_entidad`, `codigo_entidad`, `nombre_entidad`, `documento_proveedor`, `codigo_proveedor`, `proveedor_adjudicado`, `nit_del_proveedor_adjudicado`, nombres y documentos de representante legal / supervisor / ordenadores, `c_digo_bpin`, datos bancarios.

**Fuga de información** (se conocen o se actualizan *después* de adjudicar):

| Columna | Por qué hay fuga |
|---|---|
| `valor_del_contrato` | incluye las adiciones: **es** el objetivo |
| `presupuesto_general_de_la_nacion_pgn`, `sistema_general_de_participaciones`, `sistema_general_de_regal_as`, `recursos_propios*`, `recursos_de_credito` | suman el valor **final** del contrato (se verifica abajo) |
| `fecha_de_fin_del_contrato`, `duraci_n_del_contrato` | se mueven con las prórrogas (se verifica abajo); `fecha_de_fin_del_contrato` solo se usa para el tope legal de la sección 4.1 |
| `estado_contrato` | «Modificado» delata la adición |
| `dias_adicionados`, `fecha_de_notificaci_n_de_prorrogaci_n` | prórrogas posteriores |
| `valor_facturado`, `valor_pagado`, `valor_pendiente_*`, `valor_amortizado`, `saldo_cdp`, `saldo_vigencia` | ejecución financiera |
| `estado_del_procedimiento`, `estado_resumen`, `fase`, `ultima_actualizacion`, fechas de liquidación | estados posteriores |
| cláusulas en `EXCLUIR_POR_DERIVA` | cambian entre contratos nuevos y viejos (sección 4.3) |

**Texto libre** (`objeto_del_contrato`, `descripcion_del_proceso`): no es fuga, pero queda fuera de este modelo base.

In [ ]:
fuentes = sum(num(df[c]).fillna(0) for c in ["presupuesto_general_de_la_nacion_pgn", "recursos_propios"])
dur_fechas = (pd.to_datetime(df.fecha_de_fin_del_contrato, errors="coerce")
              - pd.to_datetime(df.fecha_de_inicio_del_contrato, errors="coerce")).dt.days

print("Sobrecosto según estado_contrato (fuga evidente):")
display(df.groupby("estado_contrato").sobrecosto.mean().round(3).to_frame("tasa de sobrecosto"))
print("Duración mediana según fechas del contrato (días):")
display(dur_fechas.groupby(df.sobrecosto).median().rename({0: "sin sobrecosto", 1: "con sobrecosto"}))
print(f"dias_adicionados > 0: tasa de sobrecosto {df.sobrecosto[num(df.dias_adicionados) > 0].mean():.1%} "
      f"vs {df.sobrecosto[~(num(df.dias_adicionados) > 0)].mean():.1%}")

### 5.2 Variables del modelo (disponibles al adjudicar)

* **Entidad:** departamento, orden, sector, rama, centralizada/descentralizada.
* **Contrato y proceso:** tipo de contrato, modalidad, categoría UNSPSC (segmento y familia), origen de recursos, destino del gasto, cláusulas validadas en la sección 4.3, mes de firma.
* **Montos:** log del precio base, log del valor adjudicado y **descuento** (adjudicado / precio base).
* **Duración pactada** en días (del proceso, no de las fechas del contrato).
* **Competencia:** número de lotes, proveedores invitados, respuestas recibidas, proveedores únicos que respondieron.
* **Proveedor:** pyme, grupo/consorcio, tipo de documento.

In [ ]:
CAT_CONTRATO = ["departamento", "orden", "sector", "rama", "entidad_centralizada", "tipo_de_contrato",
                "modalidad_de_contratacion", "origen_de_los_recursos", "destino_gasto", "es_pyme",
                "es_grupo", "tipodocproveedor"] + [c for c in CLAUSULAS if c not in EXCLUIR_POR_DERIVA]
CATEGORICAS = CAT_CONTRATO + ["unspsc_segmento", "unspsc_familia"]
NUMERICAS = ["log_precio_base", "log_valor_adjudicado", "adjudicado_sobre_base", "duracion_dias",
             "numero_de_lotes", "proveedores_invitados", "proveedores_con_invitacion",
             "respuestas_al_procedimiento", "proveedores_unicos_con", "mes_firma"]
DIAS = {"día(s)": 1, "Dia(s)": 1, "Semana(s)": 7, "Mes(es)": 30, "Año(s)": 365, "Hora(s)": 1 / 24}

def construir_X(d):
    X = d[CAT_CONTRATO].astype(str).copy()
    unspsc = d.codigo_de_categoria_principal.str.extract(r"V1\.(\d{8})")[0]
    X["unspsc_segmento"] = unspsc.str[:2].fillna("NA")
    X["unspsc_familia"] = unspsc.str[:4].fillna("NA")
    X["log_precio_base"] = np.log1p(d.precio_base_n)
    X["log_valor_adjudicado"] = np.log1p(d.valor_adjudicado)
    X["adjudicado_sobre_base"] = d.valor_adjudicado / d.precio_base_n
    X["duracion_dias"] = num(d.duracion) * d.unidad_de_duracion.map(DIAS)
    for c in ["numero_de_lotes", "proveedores_invitados", "proveedores_con_invitacion",
              "respuestas_al_procedimiento", "proveedores_unicos_con"]:
        X[c] = num(d[c])
    X["mes_firma"] = pd.to_datetime(d.fecha_de_firma, errors="coerce").dt.month
    return X[CATEGORICAS + NUMERICAS]

X = construir_X(df)
y = df.sobrecosto.values
grupos = df.nit_entidad.values   # solo para separar por entidad
print(X.shape, "·", len(CATEGORICAS), "categóricas ·", len(NUMERICAS), "numéricas")
X.head()

## 6. Partición por entidad y validación cruzada agrupada

Una misma entidad contrata de forma parecida; si sus contratos quedan a ambos lados de una partición, el modelo «reconoce» a la entidad y la métrica se infla. Por eso **todas** las particiones del cuaderno separan por entidad:

* **Prueba** (25 % de las entidades, `GroupShuffleSplit`): se usa **una sola vez**, en la sección 10.
* **Validación cruzada** dentro de entrenamiento (`GroupKFold`, `N_PLIEGUES` pliegues): elige hiperparámetros, el número de iteraciones del boosting, las épocas de la red y el **umbral** de cada modelo, con predicciones fuera de pliegue.

La parada temprana interna de `HistGradientBoostingClassifier` y de `MLPClassifier` separa la validación **por filas**, mezclando contratos de una misma entidad. Por eso se desactiva y el número de iteraciones o épocas se elige con estos mismos pliegues por entidad.

In [ ]:
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEMILLA).split(X, y, grupos))
X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y[tr], y[te]
g_tr, g_te = grupos[tr], grupos[te]
print(f"Entrenamiento: {len(tr):,} contratos, {len(set(g_tr)):,} entidades | tasa {y_tr.mean():.1%}")
print(f"Prueba:        {len(te):,} contratos, {len(set(g_te)):,} entidades | tasa {y_te.mean():.1%}")
assert not set(g_tr) & set(g_te)

PLIEGUES = list(GroupKFold(n_splits=N_PLIEGUES).split(X_tr, y_tr, g_tr))
for k, (a, v) in enumerate(PLIEGUES, 1):
    assert not set(g_tr[a]) & set(g_tr[v])
    print(f"  pliegue {k}: validación {len(v):,} contratos, {len(set(g_tr[v])):,} entidades, tasa {y_tr[v].mean():.1%}")

## 7. Marco de evaluación

**Métrica para elegir modelos e hiperparámetros:** PR-AUC (precisión promedio), media de los pliegues por entidad. Con 25 % de positivos, un clasificador al azar tiene PR-AUC igual a la tasa de positivos; el ROC-AUC se reporta como complemento.

**Regla del umbral (igual para todos los modelos):** el umbral que maximiza **F-beta** en las predicciones **fuera de pliegue** del entrenamiento. Con `BETA = 1` es el F1; con `BETA = 2`, la exhaustividad pesa el doble, útil si a una contraloría le cuesta más dejar pasar un sobrecosto que revisar un contrato de más. El umbral nunca se elige con la prueba ni con predicciones del propio entrenamiento.

**Incertidumbre:** bootstrap **por entidad** sobre la prueba: se remuestrean entidades completas, no filas, porque los contratos de una entidad no son independientes.

**Preprocesamiento** (ajustado siempre dentro de cada pliegue):

* *lineal* (logística y red): one-hot de categorías con al menos 30 casos, mediana para faltantes y estandarización.
* *árboles* (árbol y bosque): códigos ordinales de categoría y mediana para faltantes; los árboles no necesitan escalar.
* *boosting*: categorías nativas de `HistGradientBoostingClassifier`.

In [ ]:
prep_lineal = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), CATEGORICAS),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), NUMERICAS),
])
prep_arbol = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                           encoded_missing_value=-2), CATEGORICAS),
    ("num", SimpleImputer(strategy="median"), NUMERICAS),
])

# niveles para el boosting, aprendidos SOLO en entrenamiento (máx. 250 por columna); lo demás -> "OTRA"
NIVELES = {c: X_tr[c].value_counts().index[:250].tolist() for c in CATEGORICAS}
def a_categorias(D):
    D = D.copy()
    for c in CATEGORICAS:
        D[c] = pd.Categorical(D[c].where(D[c].isin(NIVELES[c]), "OTRA"), categories=NIVELES[c] + ["OTRA"])
    return D
Xc_tr, Xc_te = a_categorias(X_tr), a_categorias(X_te)


def fuera_de_pliegue(estimador, Xd, yd, pliegues=PLIEGUES):
    """Predicciones fuera de pliegue y PR-AUC de cada pliegue."""
    p = np.full(len(yd), np.nan)
    ap = []
    for a, v in pliegues:
        m = clone(estimador).fit(Xd.iloc[a], yd[a])
        p[v] = m.predict_proba(Xd.iloc[v])[:, 1]
        ap.append(average_precision_score(yd[v], p[v]))
    return p, np.array(ap)


def buscar(estimador, grilla, Xd=None, yd=None, regla="mejor"):
    """Prueba cada combinación con los pliegues por entidad.
    regla='mejor': mayor PR-AUC media; regla='1EE': la más simple (última de la grilla) dentro de un error estándar."""
    Xd = X_tr if Xd is None else Xd
    yd = y_tr if yd is None else yd
    filas, oofs = [], []
    for params in grilla:
        p, ap = fuera_de_pliegue(clone(estimador).set_params(**params), Xd, yd)
        filas.append({**{k.split("__")[-1]: v for k, v in params.items()},
                      "PR-AUC media": ap.mean(), "error estándar": ap.std(ddof=1) / np.sqrt(len(ap))})
        oofs.append(p)
    tabla = pd.DataFrame(filas)
    i_mejor = int(tabla["PR-AUC media"].idxmax())
    if regla == "1EE":
        piso = tabla.loc[i_mejor, "PR-AUC media"] - tabla.loc[i_mejor, "error estándar"]
        i_mejor = int(tabla.index[tabla["PR-AUC media"] >= piso].max())
    tabla["elegido"] = np.where(tabla.index == i_mejor, "◀", "")
    display(tabla.round({"PR-AUC media": 4, "error estándar": 4}))
    return grilla[i_mejor], oofs[i_mejor]


def elegir_umbral(yd, p, beta=BETA):
    prec, rec, umb = precision_recall_curve(yd, p)
    fb = (1 + beta**2) * prec[:-1] * rec[:-1] / (beta**2 * prec[:-1] + rec[:-1] + 1e-12)
    return float(umb[np.nanargmax(fb)])


MODELOS = {}   # nombre -> dict(p_te, umbral, p_oof, predictor, X_prueba)

def registrar(nombre, p_oof, p_te, predictor, X_prueba):
    u = elegir_umbral(y_tr, p_oof)
    MODELOS[nombre] = dict(p_oof=p_oof, p_te=p_te, umbral=u, predictor=predictor, X_prueba=X_prueba)
    print(f"{nombre}: PR-AUC fuera de pliegue {average_precision_score(y_tr, p_oof):.4f} · umbral F{BETA:g} = {u:.3f}")

## 8. Modelos

### 8.1 Línea base y regresión logística

La logística se ajusta con `class_weight="balanced"`, así que sus probabilidades **no** están calibradas (se reparten como si las clases fueran 50/50). Eso no afecta el orden (ROC-AUC, PR-AUC), pero hace que su umbral natural no sea 0,5: por eso su umbral también se elige con la regla F-beta fuera de pliegue. Se busca la fuerza de regularización `C`.

In [ ]:
base = DummyClassifier(strategy="prior")
p_oof_base, _ = fuera_de_pliegue(base, X_tr, y_tr)
base_final = clone(base).fit(X_tr, y_tr)
MODELOS["Línea base"] = dict(p_oof=p_oof_base, p_te=base_final.predict_proba(X_te)[:, 1], umbral=1.0,
                             predictor=base_final, X_prueba=X_te)

logit = Pipeline([("prep", prep_lineal),
                  ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))])
params_logit, oof_logit = buscar(logit, [{"clf__C": c} for c in [0.01, 0.1, 1.0]])
logit_final = clone(logit).set_params(**params_logit).fit(X_tr, y_tr)
registrar("Regresión logística", oof_logit, logit_final.predict_proba(X_te)[:, 1], logit_final, X_te)

### 8.2 Árbol de decisión

Se poda con `ccp_alpha` y la **regla de un error estándar**: entre los valores cuya PR-AUC media queda a menos de un error estándar (desviación entre pliegues dividida por √k) del mejor, se elige el más podado. `min_samples_leaf = 50` evita hojas con probabilidades inestables.

In [ ]:
arbol = Pipeline([("prep", prep_arbol),
                  ("clf", DecisionTreeClassifier(min_samples_leaf=50, random_state=SEMILLA))])
ALFAS = [0.0, 2e-5, 5e-5, 1e-4, 2e-4, 5e-4, 1e-3]
params_arbol, oof_arbol = buscar(arbol, [{"clf__ccp_alpha": a} for a in ALFAS], regla="1EE")
arbol_final = clone(arbol).set_params(**params_arbol).fit(X_tr, y_tr)
print(f"Árbol final: {arbol_final[-1].get_n_leaves()} hojas, profundidad {arbol_final[-1].get_depth()}")
registrar("Árbol de decisión", oof_arbol, arbol_final.predict_proba(X_te)[:, 1], arbol_final, X_te)

### 8.3 Bosque aleatorio

Cada árbol ve una muestra bootstrap de filas y, en cada corte, un subconjunto de variables (`max_features`). El error *out-of-bag* **no** se usa: las filas fuera de bolsa comparten entidad con las de dentro, así que sería optimista; se usa la misma validación cruzada por entidad.

In [ ]:
bosque = Pipeline([("prep", prep_arbol),
                   ("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEMILLA))])
grilla_bosque = [{"clf__max_features": mf, "clf__min_samples_leaf": hoja}
                 for mf in ["sqrt", 0.3] for hoja in [5, 20]]
params_bosque, oof_bosque = buscar(bosque, grilla_bosque)
bosque_final = clone(bosque).set_params(**params_bosque, clf__n_estimators=400).fit(X_tr, y_tr)
registrar("Bosque aleatorio", oof_bosque, bosque_final.predict_proba(X_te)[:, 1], bosque_final, X_te)

### 8.4 Gradient Boosting

Se desactiva la parada temprana interna y, para cada combinación, se entrena hasta `ITER_MAX` iteraciones en cada pliegue, midiendo la PR-AUC de validación cada `PASO` iteraciones con `staged_predict_proba`. Se elige la combinación y el número de iteraciones con mejor PR-AUC media, y el modelo final se reentrena con todo el entrenamiento y ese número fijo de iteraciones.

In [ ]:
ITER_MAX, PASO = 600, 10
gb_base = dict(learning_rate=0.06, l2_regularization=1.0, categorical_features="from_dtype",
               early_stopping=False, random_state=SEMILLA)

def curva_gb(max_leaf_nodes):
    curvas, staged = [], []
    for a, v in PLIEGUES:
        m = HistGradientBoostingClassifier(max_iter=ITER_MAX, max_leaf_nodes=max_leaf_nodes, **gb_base)
        m.fit(Xc_tr.iloc[a], y_tr[a])
        ps = [pr[:, 1] for k, pr in enumerate(m.staged_predict_proba(Xc_tr.iloc[v]), 1) if k % PASO == 0]
        curvas.append([average_precision_score(y_tr[v], p) for p in ps])
        staged.append(ps)
    return np.array(curvas), staged

filas, mejor = [], None
fig, ax = plt.subplots(figsize=(7, 3.2))
for hojas in [15, 31]:
    curvas, staged = curva_gb(hojas)
    media = curvas.mean(axis=0)
    i = int(np.argmax(media))
    iters = (np.arange(len(media)) + 1) * PASO
    ax.plot(iters, media, label=f"max_leaf_nodes={hojas}")
    filas.append({"max_leaf_nodes": hojas, "iteraciones": int(iters[i]), "PR-AUC media": media[i],
                  "error estándar": curvas[:, i].std(ddof=1) / np.sqrt(len(PLIEGUES))})
    if mejor is None or media[i] > mejor["ap"]:
        oof = np.full(len(y_tr), np.nan)
        for (a, v), ps in zip(PLIEGUES, staged):
            oof[v] = ps[i]
        mejor = dict(ap=media[i], hojas=hojas, iters=int(iters[i]), oof=oof)
ax.set(xlabel="iteraciones", ylabel="PR-AUC media (validación por entidad)", title="Gradient Boosting: elección de iteraciones")
ax.legend(); plt.tight_layout(); plt.show()
display(pd.DataFrame(filas).round(4))

gb = HistGradientBoostingClassifier(max_iter=mejor["iters"], max_leaf_nodes=mejor["hojas"], **gb_base).fit(Xc_tr, y_tr)
p_gb = gb.predict_proba(Xc_te)[:, 1]
registrar("Gradient Boosting", mejor["oof"], p_gb, gb, Xc_te)

## 9. Red neuronal (perceptrón multicapa)

### 9.1 Qué hay que decidir y por qué

Una red no aprende su propia forma: la arquitectura, la regularización y la duración del entrenamiento se fijan desde afuera y se eligen con validación. Para este problema:

| Decisión | Elección | Razón |
|---|---|---|
| Entradas | one-hot + numéricas estandarizadas (`prep_lineal`) | la red combina las entradas con pesos; sin escalar, las variables de mayor magnitud dominan el gradiente |
| Arquitectura | `(32,)` y `(64, 32)` | con datos tabulares, una o dos capas suelen bastar; se prueba una pequeña y una mediana |
| Activación | ReLU (por defecto) | el gradiente no se encoge al pasar por las capas activas |
| Regularización L2 (`alpha`) | 1e-4 y 1e-2 | controla el tamaño de los pesos; frena la memorización |
| Optimizador | Adam, tasa de aprendizaje 1e-3, lotes de 256 | valores estándar y estables |
| Épocas | elegidas con validación **por entidad** | la parada temprana de `MLPClassifier` separa por filas; aquí se entrena época por época con `partial_fit` y se mide la PR-AUC en entidades no vistas |
| Semillas | 3 semillas en el modelo final | los pesos iniciales son aleatorios; la variación entre semillas mide cuánto del resultado es azar |

Para no multiplicar el tiempo, la búsqueda usa los primeros 3 pliegues; las predicciones fuera de pliegue para el umbral usan los 5.

### 9.2 Calibración de las probabilidades

Una red entrenada con entropía cruzada y sin ponderar clases tiende a dar probabilidades razonablemente calibradas, pero conviene verificarlo: la sección 10 compara las curvas de calibración y el puntaje de Brier de todos los modelos. Si la red quedara mal calibrada y se necesitaran probabilidades (por ejemplo, para un umbral por costos), se recalibraría con `CalibratedClassifierCV` usando los mismos pliegues por entidad.

In [ ]:
ARQUITECTURAS = [(32,), (64, 32)]
ALPHAS = [1e-4, 1e-2]
EPOCAS_MAX = 30
PLIEGUES_RED = PLIEGUES[:3]

def red_nueva(arq, alpha, semilla=SEMILLA):
    return MLPClassifier(hidden_layer_sizes=arq, alpha=alpha, learning_rate_init=1e-3,
                         batch_size=256, random_state=semilla)

def entrenar_por_epocas(red, Xa, ya, epocas, Xv=None, yv=None):
    aps = []
    for _ in range(epocas):
        red.partial_fit(Xa, ya, classes=np.array([0, 1]))
        if Xv is not None:
            aps.append(average_precision_score(yv, red.predict_proba(Xv)[:, 1]))
    return red, aps

t0 = time.time()
datos_red = []
for a, v in PLIEGUES_RED:                       # el preprocesamiento se ajusta dentro de cada pliegue
    prep = clone(prep_lineal).fit(X_tr.iloc[a])
    datos_red.append((prep.transform(X_tr.iloc[a]), y_tr[a], prep.transform(X_tr.iloc[v]), y_tr[v]))

filas, curvas_red = [], {}
for arq in ARQUITECTURAS:
    for alpha in ALPHAS:
        c = np.array([entrenar_por_epocas(red_nueva(arq, alpha), Xa, ya, EPOCAS_MAX, Xv, yv)[1]
                      for Xa, ya, Xv, yv in datos_red])
        curvas_red[(arq, alpha)] = c
        media = c.mean(axis=0); i = int(np.argmax(media))
        filas.append({"arquitectura": str(arq), "alpha": alpha, "mejor época": i + 1, "PR-AUC media": media[i],
                      "error estándar": c[:, i].std(ddof=1) / np.sqrt(len(c)), "PR-AUC última época": media[-1],
                      "_clave": (arq, alpha)})
tabla_red = pd.DataFrame(filas)
display(tabla_red.drop(columns="_clave").round(4))
print(f"Búsqueda de la red: {time.time() - t0:.0f} s")

fig, ax = plt.subplots(figsize=(7, 3.2))
for (arq, alpha), c in curvas_red.items():
    ax.plot(np.arange(1, EPOCAS_MAX + 1), c.mean(axis=0), label=f"{arq}, alpha={alpha:g}")
ax.set(xlabel="época", ylabel="PR-AUC media (validación por entidad)", title="Red neuronal: épocas y configuración")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
fila = tabla_red.loc[tabla_red["PR-AUC media"].idxmax()]
(ARQ, ALPHA), EPOCAS = fila["_clave"], int(fila["mejor época"])
print(f"Red elegida: capas {ARQ}, alpha={ALPHA:g}, {EPOCAS} épocas")

# predicciones fuera de pliegue (los 5 pliegues) para el umbral
oof_red = np.full(len(y_tr), np.nan)
for a, v in PLIEGUES:
    prep = clone(prep_lineal).fit(X_tr.iloc[a])
    red, _ = entrenar_por_epocas(red_nueva(ARQ, ALPHA), prep.transform(X_tr.iloc[a]), y_tr[a], EPOCAS)
    oof_red[v] = red.predict_proba(prep.transform(X_tr.iloc[v]))[:, 1]

# modelo final con todo el entrenamiento y 3 semillas
prep_red = clone(prep_lineal).fit(X_tr)
Z_tr, Z_te = prep_red.transform(X_tr), prep_red.transform(X_te)
redes = {s: entrenar_por_epocas(red_nueva(ARQ, ALPHA, s), Z_tr, y_tr, EPOCAS)[0] for s in [SEMILLA, SEMILLA + 1, SEMILLA + 2]}
ap_semillas = {s: average_precision_score(y_te, r.predict_proba(Z_te)[:, 1]) for s, r in redes.items()}
red_final = redes[SEMILLA]
n_param = sum(w.size for w in red_final.coefs_) + sum(b.size for b in red_final.intercepts_)
print(f"Entradas tras one-hot: {Z_tr.shape[1]} · parámetros de la red: {n_param:,}")
print("PR-AUC en prueba según la semilla:", {s: round(v, 4) for s, v in ap_semillas.items()},
      f"· desviación {np.std(list(ap_semillas.values()), ddof=1):.4f}")

red_pipe = Pipeline([("prep", prep_red), ("clf", red_final)])   # pasos ya ajustados
registrar("Red neuronal", oof_red, red_final.predict_proba(Z_te)[:, 1], red_pipe, X_te)

## 10. Comparación en el conjunto de prueba

Cada modelo usa el umbral elegido con sus predicciones fuera de pliegue. El **modelo de referencia** para comparar se fija con la PR-AUC de validación cruzada (no con la prueba), y las diferencias se miden con bootstrap **pareado por entidad**: en cada remuestreo se calculan las métricas de todos los modelos sobre las mismas entidades.

In [ ]:
filas = []
for nombre, m in MODELOS.items():
    p, u = m["p_te"], m["umbral"]
    pred = p >= u
    filas.append({"modelo": nombre,
                  "PR-AUC CV": average_precision_score(y_tr, m["p_oof"]),
                  "ROC-AUC": roc_auc_score(y_te, p), "PR-AUC": average_precision_score(y_te, p),
                  "umbral": u, "precisión": precision_score(y_te, pred, zero_division=0),
                  "exhaustividad": recall_score(y_te, pred), "F1": f1_score(y_te, pred),
                  "% marcados": pred.mean(), "Brier": brier_score_loss(y_te, p)})
tabla = pd.DataFrame(filas).set_index("modelo")
REFERENCIA = tabla.drop(index="Línea base")["PR-AUC CV"].idxmax()
print(f"Modelo de referencia (mejor PR-AUC en validación cruzada): {REFERENCIA}")
display(tabla.round(3))

# bootstrap pareado por entidad
rng = np.random.default_rng(SEMILLA)
pos_por_entidad = pd.Series(np.arange(len(y_te))).groupby(g_te).indices
entidades = np.array(list(pos_por_entidad))
boot = {n: {"ap": [], "auc": []} for n in MODELOS}
for _ in range(N_BOOT):
    ii = np.concatenate([pos_por_entidad[e] for e in rng.choice(entidades, len(entidades), replace=True)])
    if y_te[ii].min() == y_te[ii].max():
        continue
    for n, m in MODELOS.items():
        boot[n]["ap"].append(average_precision_score(y_te[ii], m["p_te"][ii]))
        boot[n]["auc"].append(roc_auc_score(y_te[ii], m["p_te"][ii]))
boot = {n: {k: np.array(v) for k, v in d.items()} for n, d in boot.items()}

ic = pd.DataFrame({n: {"PR-AUC": f"{np.median(d['ap']):.3f} [{np.percentile(d['ap'], 2.5):.3f}, {np.percentile(d['ap'], 97.5):.3f}]",
                       "ROC-AUC": f"{np.median(d['auc']):.3f} [{np.percentile(d['auc'], 2.5):.3f}, {np.percentile(d['auc'], 97.5):.3f}]",
                       f"Δ PR-AUC vs {REFERENCIA}": "—" if n == REFERENCIA else
                           f"{np.median(d['ap'] - boot[REFERENCIA]['ap']):+.3f} "
                           f"[{np.percentile(d['ap'] - boot[REFERENCIA]['ap'], 2.5):+.3f}, "
                           f"{np.percentile(d['ap'] - boot[REFERENCIA]['ap'], 97.5):+.3f}]"}
                   for n, d in boot.items()}).T
print(f"Intervalos de 95 % (bootstrap por entidad, {len(boot[REFERENCIA]['ap'])} remuestreos):")
display(ic)
print("Una diferencia cuyo intervalo no contiene 0 es distinguible del azar de qué entidades tocan en la prueba.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for nombre, m in MODELOS.items():
    if nombre == "Línea base":
        continue
    prec, rec, _ = precision_recall_curve(y_te, m["p_te"])
    axes[0].plot(rec, prec, label=nombre)
    frac, media = calibration_curve(y_te, m["p_te"], n_bins=10, strategy="quantile")
    axes[1].plot(media, frac, marker="o", ms=3, label=nombre)
axes[0].axhline(y_te.mean(), color="gray", ls="--", label="azar")
axes[0].set(xlabel="exhaustividad", ylabel="precisión", title="Curvas precisión-exhaustividad (prueba)")
axes[1].plot([0, 1], [0, 1], color="gray", ls="--")
axes[1].set(xlabel="probabilidad predicha", ylabel="frecuencia observada", title="Calibración (prueba)")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

m = MODELOS[REFERENCIA]
print(f"Matriz de confusión de {REFERENCIA} con umbral {m['umbral']:.3f} [filas = real, columnas = predicho]:")
print(confusion_matrix(y_te, m["p_te"] >= m["umbral"]))

In [ ]:
print(f"Desempeño de {REFERENCIA} por modalidad (prueba, modalidades con al menos 200 contratos):")
mod_te = df.modalidad_de_contratacion.values[te]
filas = []
for mod in pd.unique(mod_te):
    s = mod_te == mod
    if s.sum() >= 200 and 0 < y_te[s].mean() < 1:
        filas.append({"modalidad": mod, "contratos": int(s.sum()), "tasa": y_te[s].mean(),
                      "PR-AUC": average_precision_score(y_te[s], m["p_te"][s]),
                      "ROC-AUC": roc_auc_score(y_te[s], m["p_te"][s])})
display(pd.DataFrame(filas).sort_values("contratos", ascending=False).set_index("modalidad").round(3))
print("Si el ROC-AUC dentro de cada modalidad es mucho menor que el global, parte del desempeño viene de "
      "distinguir modalidades con tasas distintas, no de ordenar contratos dentro de cada una.")

## 11. Importancia de variables (permutación sobre el conjunto de prueba)

Se permuta cada variable y se mide cuánto cae la PR-AUC del modelo de referencia. Mide cuánto **depende este modelo** de cada variable, no si la variable causa el sobrecosto; con variables correlacionadas (precio base y valor adjudicado), la importancia se reparte entre ellas.

In [ ]:
m = MODELOS[REFERENCIA]
imp = permutation_importance(m["predictor"], m["X_prueba"], y_te, scoring="average_precision",
                             n_repeats=5, random_state=SEMILLA, n_jobs=-1)
importancia = pd.Series(imp.importances_mean, index=m["X_prueba"].columns).sort_values()
ax = importancia.tail(15).plot.barh(figsize=(7, 5), color="#3b6ea5",
                                    xerr=pd.Series(imp.importances_std, index=m["X_prueba"].columns)[importancia.tail(15).index])
ax.set(title=f"{REFERENCIA}: caída de PR-AUC al permutar cada variable", xlabel="Δ PR-AUC")
plt.tight_layout(); plt.show()
TOP_PRINCIPAL = importancia.sort_values(ascending=False).head(5).index.tolist()
print("Cinco variables más importantes:", TOP_PRINCIPAL)

## 12. Robustez: ¿las conclusiones dependen del corte del objetivo?

Se repite, en ambos escenarios de la sección 4.1, una versión rápida del análisis: misma partición por entidad, logística y Gradient Boosting con los hiperparámetros ya elegidos, y las cinco variables más importantes del boosting. Si el orden de los modelos y las variables principales coinciden, la conclusión no depende de cómo se limpió el objetivo.

In [ ]:
def ajuste_rapido(d):
    Xa, ya, ga = construir_X(d), d.sobrecosto.values, d.nit_entidad.values
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEMILLA).split(Xa, ya, ga))
    lg = clone(logit).set_params(**params_logit).fit(Xa.iloc[a], ya[a])
    niveles = {c: Xa.iloc[a][c].value_counts().index[:250].tolist() for c in CATEGORICAS}
    conv = lambda D: D.assign(**{c: pd.Categorical(D[c].where(D[c].isin(niveles[c]), "OTRA"),
                                                     categories=niveles[c] + ["OTRA"]) for c in CATEGORICAS})
    g = HistGradientBoostingClassifier(max_iter=mejor["iters"], max_leaf_nodes=mejor["hojas"], **gb_base)
    g.fit(conv(Xa.iloc[a]), ya[a])
    p_lg, p_g = lg.predict_proba(Xa.iloc[b])[:, 1], g.predict_proba(conv(Xa.iloc[b]))[:, 1]
    imp = permutation_importance(g, conv(Xa.iloc[b]), ya[b], scoring="average_precision",
                                 n_repeats=3, random_state=SEMILLA, n_jobs=-1)
    top = pd.Series(imp.importances_mean, index=Xa.columns).sort_values(ascending=False).head(5).index.tolist()
    return {"contratos": len(d), "tasa": round(ya.mean(), 3),
            "PR-AUC logística": round(average_precision_score(ya[b], p_lg), 3),
            "PR-AUC boosting": round(average_precision_score(ya[b], p_g), 3),
            "ROC-AUC logística": round(roc_auc_score(ya[b], p_lg), 3),
            "ROC-AUC boosting": round(roc_auc_score(ya[b], p_g), 3), "top 5 (boosting)": top}

robustez = pd.DataFrame({nombre: ajuste_rapido(d) for nombre, d in ESCENARIOS.items()}).T
display(robustez)
comunes = set(robustez.loc["nominal", "top 5 (boosting)"]) & set(robustez.loc["legal", "top 5 (boosting)"])
print(f"Variables en el top 5 de ambos escenarios: {len(comunes)} de 5 -> {sorted(comunes)}")

## 13. ¿Cuánto infla la fuga de información las métricas?

Mismo boosting y misma partición, agregando columnas con fuga. Estas cifras **no son válidas**: se muestran para justificar la exclusión.

In [ ]:
def evaluar(nombre, p, umbral):
    return {"modelo": nombre, "ROC-AUC": roc_auc_score(y_te, p),
            "PR-AUC": average_precision_score(y_te, p), "F1": f1_score(y_te, p >= umbral)}

def con_extras(extras):
    A, B = Xc_tr.copy(), Xc_te.copy()
    for nombre, serie in extras.items():
        A[nombre], B[nombre] = serie.iloc[tr].values, serie.iloc[te].values
    mod = HistGradientBoostingClassifier(max_iter=mejor["iters"], max_leaf_nodes=mejor["hojas"], **gb_base).fit(A, y_tr)
    return mod.predict_proba(B)[:, 1]

u_gb = MODELOS["Gradient Boosting"]["umbral"]
fuga = {
    "estado_contrato": df.estado_contrato.astype("category"),
    "dias_adicionados": num(df.dias_adicionados),
    "fuente_pgn": num(df.presupuesto_general_de_la_nacion_pgn),
    "fuente_recursos_propios": num(df.recursos_propios),
    "duracion_por_fechas": dur_fechas,
}
comparacion = pd.DataFrame([
    evaluar("Limpio (al adjudicar)", p_gb, u_gb),
    evaluar("+ fuga (estado, días adicionados, fuentes, fechas)", con_extras(fuga), u_gb),
    evaluar("+ fuga total (valor_del_contrato)", con_extras({**fuga, "log_valor_final": np.log1p(df.valor_final)}), u_gb),
]).set_index("modelo").round(3)
comparacion

## 14. Conclusiones

In [ ]:
t = tabla.drop(index="Línea base")
d_ref = {n: boot[n]["ap"] - boot[REFERENCIA]["ap"] for n in MODELOS if n not in ("Línea base", REFERENCIA)}
empatados = [n for n, d in d_ref.items() if np.percentile(d, 2.5) <= 0 <= np.percentile(d, 97.5)]
print(f"Escenario del objetivo: {CORTE_PRINCIPAL} · {len(df):,} contratos · tasa de sobrecosto {df.sobrecosto.mean():.1%}")
print(f"Mejor modelo en validación cruzada: {REFERENCIA} (PR-AUC CV {t.loc[REFERENCIA, 'PR-AUC CV']:.3f}; "
      f"prueba: PR-AUC {t.loc[REFERENCIA, 'PR-AUC']:.3f}, ROC-AUC {t.loc[REFERENCIA, 'ROC-AUC']:.3f}; "
      f"línea base PR-AUC {tabla.loc['Línea base', 'PR-AUC']:.3f})")
print(f"Con su umbral: precisión {t.loc[REFERENCIA, 'precisión']:.0%}, exhaustividad {t.loc[REFERENCIA, 'exhaustividad']:.0%}, "
      f"marca al {t.loc[REFERENCIA, '% marcados']:.0%} de los contratos")
print(f"Modelos no distinguibles de {REFERENCIA} en PR-AUC (bootstrap por entidad): {empatados or 'ninguno'}")
print(f"Variables principales: {TOP_PRINCIPAL}")
print(f"Robustez al corte del objetivo: {len(comunes)} de 5 variables principales coinciden entre escenarios")

* La llave entre las dos tablas es `proceso_de_compra` ↔ `id_del_portafolio`, y la variable objetivo es `valor_del_contrato > valor_total_adjudicacion`; la sección 4.2 contrasta esa definición con el conjunto de adiciones.
* El tope legal de adiciones se mide en salarios mínimos: un corte fijo de 1,5x en pesos descarta contratos legales que se adicionaron en años con salario mínimo más alto. La sección 12 muestra si eso cambia las conclusiones.
* Varias columnas aparentemente inocentes tienen **fuga**: las fuentes de financiación suman el valor final, las fechas de fin se mueven con las prórrogas y algunas cláusulas pueden actualizarse después de firmar (sección 4.3). Incluirlas infla el desempeño (sección 13).
* Todas las decisiones (hiperparámetros, iteraciones, épocas y umbrales) se toman con validación cruzada **por entidad** dentro del entrenamiento; la prueba, con entidades no vistas, se usa una sola vez, y sus diferencias se leen con intervalos de bootstrap por entidad.

**Limitaciones:** algunos contratos aún pueden recibir adiciones (etiqueta incompleta); solo se usan procesos con un contrato; las modalidades sin ofertas quedan por fuera mientras no tengan un valor adjudicado comparable (sección 4.4); la cota del tope legal usa la fecha de fin del contrato porque la fecha de cada adición no está en la tabla. **Trabajo futuro:** validación temporal con varios años, el texto del objeto del contrato, el historial de la entidad calculado solo con contratos anteriores y, para la contratación directa, el valor inicial reconstruido con el conjunto de adiciones.